# Deligne–Mumford moduli spike — research highlights

Mathematician-facing tour of the owned vocabulary in `dm_moduli_spike`.

**Semantic layers (never conflated):**

| object | meaning |
| --- | --- |
| `M_gn` / `Mbar_gn` | smooth / stable moduli stacks over a base scheme |
| `StableGraph` / `StableGraphs` | labeled dual-graph representatives; contractions live here |
| `StableGraphCategory` ($\Gamma_{g,n}$) | finite combinatorial category of dual graphs |
| `Stratum` / `Stratification` | locally closed strata of $\overline{\mathcal M}_{g,n}$ indexed by dual graphs |
| `QuotientStack` / `ClutchingMorphism` | stratum presentation $[\prod_v \mathcal M_{w(v),n_v}/\operatorname{Aut}(\Gamma)]$ and clutching |

**Prerequisite:** repository `.envrc` puts `computations/experiments` on `PYTHONPATH`.

**Honesty:** axiom membership (`X in DeligneMumfordStacks(k).Proper()`) is the constructor's
theorem stamp via `declared_axioms`, not an analytic proof from equations.

In [ ]:
from sage.graphs.graph import Graph
from sage.all import QQ
from sage.categories.homset import Hom
from dm_moduli_spike import (
    M_gn,
    Mbar_gn,
    StableDualGraph,
    StableGraphCategory,
    StableGraphs,
    QuotientStack,
    ProductStack,
    Stratifications,
    spec,
)


def strata_by_edges(Sigma):
    """Group strata by number of nodes (= edges of the dual graph)."""
    buckets = {}
    for G in Sigma.index_poset():
        buckets.setdefault(G.num_edges(), []).append(G)
    for codim in sorted(buckets):
        graphs = buckets[codim]
        print(f"codimension {codim}  ({len(graphs)} type{'s' if len(graphs) != 1 else ''})")
        for G in graphs:
            print(
                f"    Γ = {G}   (|Aut| = {G.automorphism_group().order()})"
            )
        print()


def stratum_label(G):
    if G.num_edges() == 0:
        return "smooth"
    return f"e={G.num_edges()}\nw={G.vertex_genera()}"


def show_specialization_poset(Sigma, *, labeler=stratum_label, figsize=8):
    poset = Sigma.specialization_poset()
    hasse = poset.hasse_diagram()
    hasse.relabel({G: labeler(G) for G in poset}, inplace=True)
    hasse.show(graph_border=True, vertex_size=900, figsize=figsize)
    return hasse


def show_weighted_dual_graph(G, *, title=None, figsize=4):
    """Draw a weighted stable dual graph from public StableGraph APIs."""
    H = G.half_edges()
    edge_list = []
    loop_vertices = []
    seen = set()
    for edge in G.edges():
        a, b = edge.half_edges()
        u, v = H(a).vertex(), H(b).vertex()
        if u == v:
            if u not in loop_vertices:
                loop_vertices.append(u)
        else:
            key = (min(u, v), max(u, v))
            if key not in seen:
                seen.add(key)
                edge_list.append(key)
    graph = Graph(edge_list, loops=sorted(loop_vertices))
    marks_at = {v: [] for v in range(G.num_vertices())}
    for leg in G.legs():
        marks_at[leg.vertex()].append(leg.label())
    labels = {}
    genera = G.vertex_genera()
    for vertex in range(G.num_vertices()):
        text = f"v{vertex}\nw={genera[vertex]}"
        marks = marks_at[vertex]
        if marks:
            text += f"\n{tuple(marks)}"
        labels[vertex] = text
    graph.relabel(labels, inplace=True)
    print(title or str(G))
    graph.show(graph_border=True, vertex_size=500, figsize=figsize)
    return graph

## 1. Choose ambient moduli stacks

Work over a base scheme `k = spec(QQ)`. Open and compactified moduli are
`M_gn` and `Mbar_gn`; the open immersion is the compactification morphism.

In [ ]:
k = spec(QQ)
X = M_gn(1, 2, base=k)
Xbar = Mbar_gn(1, 2, base=k)
c = X.compactification()
print(X)
print(Xbar)
print(c)
assert c.codomain() is Xbar
X, Xbar, c

Dual graphs live in the Sage parent `StableGraphs(g, I)`. The combinatorial
category $\Gamma_{g,n}$ is `StableGraphCategory(g, n)`.

In [ ]:
Graphs = StableGraphs(1, (1, 2))
Gamma = StableGraphCategory(1, 2)
smooth = Graphs.smooth()
print(Graphs)
print(Gamma)
print("smooth:", smooth)
print("|Aut(smooth)| =", smooth.automorphism_group().order())
list(Graphs)

## 2. Boundary stratification

`Xbar.stratification(by=StableDualGraph())` returns a `Stratification` whose
index poset is the specialization poset on dual graphs.

In [ ]:
Sigma = Xbar.stratification(by=StableDualGraph())
assert Sigma.parent() is Stratifications(Xbar)
print(Sigma)
print("indexing category:", Sigma.indexing_category())
strata_by_edges(Sigma)

In [ ]:
P = Sigma.specialization_poset()
print(P)
print("cover relations (generic → special):")
for generic, special in P.cover_relations():
    print(f"  {generic}  ≤  {special}")

### Hasse diagrams

Genus zero $\overline{\mathcal M}_{0,4}$ is small enough to draw.

In [ ]:
X04 = Mbar_gn(0, 4, base=k)
Sigma04 = X04.stratification(by=StableDualGraph())
print("Hasse diagram for", X04)
show_specialization_poset(Sigma04)

In [ ]:
print("Hasse diagram for", Xbar)
show_specialization_poset(Sigma)

### Weighted dual graphs

In [ ]:
print("Dual graphs for strata in", X04)
for G in Sigma04.index_poset():
    show_weighted_dual_graph(G)

## 3. Contractions

Elementary contractions are morphisms in $\Gamma_{g,n}$.

In [ ]:
nodal = next(G for G in Graphs if G.num_edges() > 0)
print("nodal:", nodal)
for target, morph, _size in nodal.elementary_contractions():
    print("contract →", target)
    print("  morphism:", morph)
    print("  is_contraction:", morph.is_contraction())
    assert morph in Hom(nodal, target)

## 4. Stratum presentations and clutching

Each boundary stratum is a locally closed substack whose underlying object is
a `QuotientStack`; clutching is a stack morphism into `Mbar`.

In [ ]:
G = next(g for g in Sigma.index_poset() if g.num_edges() > 0)
S = Sigma.stratum(G)
print(S)
print("underlying:", S.underlying_stack())
assert isinstance(S.underlying_stack(), QuotientStack)
xi = S.clutching_morphism()
print("clutching:", xi)
assert xi is not None
assert xi.codomain() is Xbar
assert isinstance(xi.domain(), ProductStack)
S, xi

## 5. Genus zero specialization poset

For $g=0$ the dual-graph poset is the compatible-splits poset (including the
smooth stratum).

In [ ]:
P04 = Sigma04.specialization_poset()
print(P04)
print("|P| =", P04.cardinality())
print("boundary |P| =", X04.boundary().stratification_poset().cardinality())
assert P04.cardinality() == X04.boundary().stratification_poset().cardinality() + 1

---

**Where to go next.** Literature oracles live in `tests/literature/`; Γ category
oracles in `tests/core/test_gamma_category.sage`; geometric ontology in
`tests/core/test_geometric_ontology.sage`.